# 입력 feature 선택 및 XGBoost 모델 학습 및 검증

- feature 설정: `services/ai/configs/feature_config.yaml`
- 데이터: `services/ai/data/behavior/trial_*.json`
- 저장: `services/ai/train/model_joblib/xgboost/model.joblib`


> **트리비얼 검증 사전 (EDA 4번 결과 인용)**
>
> EDA 트리비얼 검증 (`eda_gyeom_2.ipynb` 4번 Personal EDA) 에서 high_smd_features 9개 중 다음 2개를 학습 입력 제외:
>
> - `time_to_first_click_ms` — 단독 분리 ~100%. `human_recorder` 의 첫 클릭 녹화 트리거 구조 산물. 수집 도구 변경 시 시그널 사라짐 → 진짜 매크로 시그니처 X.
> - `pre_click_mousemove_count` — 단독 분리 95%+. `lv2_collector` 의 `BEZIER_POINTS=20` 하드코딩 노출. lv3 collector 등 알고리즘 변경 시 무의미.
>
> 본 노트북은 나머지 7개 (`input_features_gyeom` yaml 그룹) 로만 학습.
>
> **단일 feature importance 50%+ 쏠리면 잠복 트리비얼 의심** — 6번 importance 다음의 사후 검증 안내 참조.

## 1. 데이터 준비


In [ ]:
from __future__ import annotations

import json
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import yaml

SEED = 42
TEST_SIZE = 0.2
MODEL_NAME = "xgboost_gyeom"
LABEL_MAPPING = {"human": 0, "macro": 1}

# 데이터 경로 설정
ROOT = Path.cwd()

if ROOT.name != "model_experiment":
    raise Exception("데이터를 찾기 위해 작업 디렉토리 확인 필요")

CONFIG_PATH = Path("../../configs/feature_config.yaml")
DATA_DIR = Path("../../data/behavior")
MODEL_DIR = Path("../model_joblib") / MODEL_NAME

print("ROOT:", ROOT)
print("CONFIG_PATH:", CONFIG_PATH)
print("DATA_DIR:", DATA_DIR)
print("MODEL_DIR:", MODEL_DIR)


In [ ]:
with CONFIG_PATH.open("r", encoding="utf-8") as f:
    feature_config = yaml.safe_load(f)

FEATURE_GROUPS = feature_config["groups"]
FEATURE_NAME_KO = feature_config.get("names_ko", {})
EXCLUDE_COLS = set(feature_config.get("exclude", []))
FEATURE_NAMES = [feature for group in FEATURE_GROUPS.values() for feature in group]

# DEVIATION (chan): yaml input_features_gyeom 그룹에서 학습 입력 7개 로드 (트리비얼 2개 제외 후 확정)
INPUT_FEATURES = feature_config["input_features_gyeom"]

def extract_label(payload: dict) -> str | None:
    label = payload.get("label")
    if isinstance(label, str):
        return label
    summary = payload.get("summary")
    if isinstance(summary, dict) and isinstance(summary.get("label"), str):
        return summary["label"]
    return None


def parse_path_pattern(value: object) -> tuple[float, float]:
    if not isinstance(value, str):
        return (np.nan, np.nan)

    parts = value.split("|", 1)
    if len(parts) != 2:
        return (np.nan, np.nan)

    distance_text = parts[0].replace("px", "").strip()
    straightness_text = parts[1].replace("straight", "").strip()

    try:
        return (float(distance_text), float(straightness_text))
    except ValueError:
        return (np.nan, np.nan)


def load_trials(data_dir: Path) -> pd.DataFrame:
    rows: list[dict] = []
    counters = {"total": 0, "kept": 0, "no_label": 0, "skipped_other": 0}

    for path in sorted(data_dir.glob("trial_*.json")):
        counters["total"] += 1
        payload = json.loads(path.read_text(encoding="utf-8"))

        label = extract_label(payload)
        if label is None:
            counters["no_label"] += 1
            continue
        if label not in LABEL_MAPPING:
            counters["skipped_other"] += 1
            continue

        summary = payload.get("summary") or {}
        metrics = payload.get("metrics") or {}
        row = {
            "trial_id": int(payload.get("trialId") or payload.get("trial_id") or 0),
            "label": label,
        }
        if isinstance(summary, dict):
            row.update(summary)
        if isinstance(metrics, dict):
            row.update(metrics)
        rows.append(row)
        counters["kept"] += 1

    df = pd.DataFrame(rows).sort_values("trial_id").reset_index(drop=True)
    print("load counters:", counters)
    return df


def add_pre_click_path_features(df: pd.DataFrame) -> None:
    for win in [300, 500]:
        raw_col = f"pre_click_mouse_path_pattern_{win}ms"
        dist_col = f"pre_click_path_{win}ms_total_distance_px"
        straight_col = f"pre_click_path_{win}ms_straightness"
        if raw_col not in df.columns or (dist_col in df.columns and straight_col in df.columns):
            continue
        parsed = df[raw_col].map(parse_path_pattern)
        df[dist_col] = parsed.map(lambda x: x[0])
        df[straight_col] = parsed.map(lambda x: x[1])


df = load_trials(DATA_DIR)
add_pre_click_path_features(df)

display(df["label"].value_counts().rename_axis("label").reset_index(name="count"))
print("configured feature count:", len(FEATURE_NAMES))
print("INPUT_FEATURES (gyeom):", len(INPUT_FEATURES))
display(pd.DataFrame({"feature": INPUT_FEATURES, "feature_ko": [FEATURE_NAME_KO.get(f, f) for f in INPUT_FEATURES]}))


## 2. 데이터 전처리


In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    log_loss,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.preprocessing import StandardScaler

present_features = [f for f in INPUT_FEATURES if f in df.columns]
missing_features = [f for f in INPUT_FEATURES if f not in df.columns]

X_all = df[present_features].apply(pd.to_numeric, errors="coerce")
usable_features = [f for f in present_features if X_all[f].notna().any()]
dropped_all_nan = [f for f in present_features if f not in usable_features]

X = X_all[usable_features]
y = df["label"].map(LABEL_MAPPING).astype(int).to_numpy()
feature_names = list(X.columns)

x_train, x_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=SEED,
    stratify=y,
)

n_neg = int((y_train == 0).sum())
n_pos = int((y_train == 1).sum())
SCALE_POS_WEIGHT = n_neg / max(n_pos, 1)

print("usable features:", len(usable_features), "/", len(INPUT_FEATURES))
print("missing features:", missing_features)
print("dropped all-NaN features:", dropped_all_nan)
print("train shape:", x_train.shape, "val shape:", x_val.shape)
print("train class counts:", dict(zip(*np.unique(y_train, return_counts=True))))
print("val class counts:", dict(zip(*np.unique(y_val, return_counts=True))))
print("scale_pos_weight:", SCALE_POS_WEIGHT)


## 3. 모델 준비


In [ ]:
from xgboost import XGBClassifier

def build_model(seed: int = SEED):
    return Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            (
                "clf",
                XGBClassifier(
                    n_estimators=400,
                    max_depth=3,
                    learning_rate=0.03,
                    subsample=0.9,
                    colsample_bytree=0.9,
                    objective="binary:logistic",
                    eval_metric="logloss",
                    scale_pos_weight=SCALE_POS_WEIGHT,
                    tree_method="hist",
                    random_state=seed,
                    n_jobs=-1,
                ),
            ),
        ]
    )


def predict_macro_proba(fitted_model, x) -> np.ndarray:
    if hasattr(fitted_model, "predict_proba"):
        return np.asarray(fitted_model.predict_proba(x))[:, 1]
    decision = np.asarray(fitted_model.decision_function(x), dtype=float)
    return 1.0 / (1.0 + np.exp(-decision))


def evaluate_binary(y_true: np.ndarray, proba: np.ndarray, threshold: float = 0.5) -> dict[str, float]:
    pred = (proba >= threshold).astype(int)
    out = {
        "accuracy": float(accuracy_score(y_true, pred)),
        "precision": float(precision_score(y_true, pred, zero_division=0)),
        "recall": float(recall_score(y_true, pred, zero_division=0)),
        "f1": float(f1_score(y_true, pred, zero_division=0)),
        "average_precision": float(average_precision_score(y_true, proba)),
    }
    try:
        out["roc_auc"] = float(roc_auc_score(y_true, proba))
    except ValueError:
        out["roc_auc"] = float("nan")
    try:
        out["log_loss"] = float(log_loss(y_true, proba, labels=[0, 1]))
    except ValueError:
        out["log_loss"] = float("nan")
    return out


model = build_model(SEED)
model


## 4. 모델 학습


In [ ]:
model.fit(x_train, y_train)

train_proba = predict_macro_proba(model, x_train)
val_proba = predict_macro_proba(model, x_val)

train_metrics = evaluate_binary(y_train, train_proba)
val_metrics = evaluate_binary(y_val, val_proba)

metrics_df = pd.DataFrame(
    [
        {"split": "train", **train_metrics},
        {"split": "validation", **val_metrics},
    ]
)
display(metrics_df)

cm = confusion_matrix(y_val, (val_proba >= 0.5).astype(int), labels=[0, 1])
cm_df = pd.DataFrame(cm, index=["actual_human", "actual_macro"], columns=["pred_human", "pred_macro"])
display(cm_df)


## 5. 모델 검증


In [ ]:
scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc",
    "average_precision": "average_precision",
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
cv_result = cross_validate(
    build_model(SEED),
    X,
    y,
    cv=cv,
    scoring=scoring,
    n_jobs=-1,
    return_train_score=False,
)

cv_summary = pd.DataFrame(
    [
        {
            "metric": metric.replace("test_", ""),
            "mean": float(values.mean()),
            "std": float(values.std(ddof=1)),
        }
        for metric, values in cv_result.items()
        if metric.startswith("test_")
    ]
).sort_values("metric")

display(cv_summary)


In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay, PrecisionRecallDisplay, RocCurveDisplay

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

ConfusionMatrixDisplay.from_predictions(
    y_val,
    (val_proba >= 0.5).astype(int),
    display_labels=["human", "macro"],
    cmap="Blues",
    colorbar=False,
    ax=axes[0],
)
axes[0].set_title("Validation Confusion Matrix")

RocCurveDisplay.from_predictions(y_val, val_proba, ax=axes[1])
axes[1].set_title("Validation ROC")

PrecisionRecallDisplay.from_predictions(y_val, val_proba, ax=axes[2])
axes[2].set_title("Validation Precision-Recall")

plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 4.5))
plt.hist(val_proba[y_val == 0], bins=20, alpha=0.7, label="human")
plt.hist(val_proba[y_val == 1], bins=20, alpha=0.7, label="macro")
plt.title("Validation P(macro) Distribution")
plt.xlabel("P(macro)")
plt.ylabel("count")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
clf = model.named_steps["clf"]
importance_df = pd.DataFrame(
    {
        "feature": feature_names,
        "feature_ko": [FEATURE_NAME_KO.get(f, f) for f in feature_names],
        "importance": clf.feature_importances_,
    }
).sort_values("importance", ascending=False)
display(importance_df.head(30))


> **트리비얼 검증 사후 — importance 결과 점검**
>
> 위 표에서 단일 feature 가 importance 의 **50%+** 를 차지하면 잠복 트리비얼 의심:
>
> 1. 해당 feature 단독 분류 정확도 측정 (`from sklearn.linear_model import LogisticRegression; cross_val_score(LogisticRegression(max_iter=1000), X_single, y, cv=5, scoring="accuracy")`)
> 2. 95%+ 면 잠복 트리비얼 확정 → `input_features_gyeom` 에서 제거 + EDA 4번 `TRIVIAL_RISK` 리스트 갱신 검토
> 3. ~50% 분담 (mouse 그룹 여러 feature 가 비슷하게 기여) 이면 정상 패턴
>
> recall < 0.85 인 경우도 importance top feature 가 트리비얼이라 모델이 그 feature 만 의존하는 상황일 수 있음.

## 6. 모델 저장


In [ ]:
import joblib

MODEL_DIR.mkdir(parents=True, exist_ok=True)
model_path = MODEL_DIR / "model.joblib"
meta_path = MODEL_DIR / "meta.json"
metrics_path = MODEL_DIR / "metrics.json"

joblib.dump(model, model_path)

meta = {
    "model_name": MODEL_NAME,
    "trained_at": datetime.now().isoformat(timespec="seconds"),
    "label_mapping": LABEL_MAPPING,
    "feature_config": str(CONFIG_PATH),
    "data_dir": str(DATA_DIR),
    "feature_names": feature_names,
    "missing_features": missing_features,
    "dropped_all_nan_features": dropped_all_nan,
    "n_samples": int(X.shape[0]),
    "n_train": int(x_train.shape[0]),
    "n_validation": int(x_val.shape[0]),
    "test_size": TEST_SIZE,
    "seed": SEED,
}
metrics_payload = {
    "train": train_metrics,
    "validation": val_metrics,
    "cv": cv_summary.to_dict(orient="records"),
    "confusion_matrix_validation": cm.astype(int).tolist(),
}

meta_path.write_text(json.dumps(meta, ensure_ascii=False, indent=2), encoding="utf-8")
metrics_path.write_text(json.dumps(metrics_payload, ensure_ascii=False, indent=2), encoding="utf-8")

print("saved model:", model_path)
print("saved meta:", meta_path)
print("saved metrics:", metrics_path)


## 7. 결과 리포트


In [ ]:
print(f"MODEL: {MODEL_NAME}")
print("Validation metrics")
display(pd.DataFrame([val_metrics]))
print("5-fold CV summary")
display(cv_summary)
print("Artifacts")
print(MODEL_DIR)
